In [ ]:
import pandas as pd
import numpy as np
import pathlib

In [ ]:
from lsff_utils import data_processing, config_utils

In [ ]:
# TODO: Deduplicate this list with the Snakefiles
data_needs = {
    "Country-Vehicle": {
        "Vehicle consumption by WRA -- any": "vehicle_consumption/any",
        'Vehicle "fortifiability" (essentially amount industrially produced)': "vehicle_consumption/fortifiability",
        "Vehicle consumption by WRA -- amount": "vehicle_consumption/amount",
    },
    "Country-Vehicle-Fort": {
        "Vehicle fortification at baseline -- any": "baseline_fortification/any_coverage",
        "Baseline effective % of fortified": "baseline_fortification/effectiveness",
        "Vehicle fortification at baseline -- amount among fortified": "baseline_fortification/concentration",
    },
    "Scenario Definition": {
        "Intervention coverage % of fortifiable": "intervention_fortification/any_coverage",
        "Intervention effective % of fortified": "intervention_fortification/effectiveness",
        "Vehicle fortification in intervention -- amount among fortified": "intervention_fortification/concentration",
    },
}

data_point_names = {
    "mean": "mean",
    "standard deviation": "sd",
}

In [ ]:
results_dir = "../results"
pathlib.Path(results_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
definition_columns_detailed_values = {
    "wealth_quintile": set(data_processing.WEALTH_QUINTILES),
    "sex": {"Female", "Male"},
}

In [ ]:
def expand(data_point_rows, definition_columns):
    for definition_column in definition_columns:
        assert set(data_point_rows[definition_column].astype(str).str.lower()) <= (
            {str(x).lower() for x in definition_columns_detailed_values[definition_column]}
            | {"all (assumed same)", "total"}
        )
        # Expand assumptions
        data_point_rows = pd.concat(
            [
                data_point_rows[
                    data_point_rows[definition_column].astype(str).str.lower()
                    != "all (assumed same)"
                ]
            ]
            + [
                data_point_rows[
                    data_point_rows[definition_column].astype(str).str.lower()
                    == "all (assumed same)"
                ].assign(
                    **{
                        definition_column: (
                            value.title() if definition_column == "sex" else value
                        )
                    }
                )
                for value in sorted(
                    list(
                        {
                            str(x)
                            for x in definition_columns_detailed_values[
                                definition_column
                            ]
                        }
                        | {"total"}
                    )
                )
            ],
            ignore_index=True,
        )
    return data_point_rows

In [ ]:
def reformat_data(data_point_rows):
    columns_to_keep = {
        "Vehicle": "vehicle_name",
        "Sex": "sex",
        "Quintile": "wealth_quintile",
        "Value": "value",
    }
    data_point_rows = data_point_rows[
        [c for c in data_point_rows.columns if c in columns_to_keep.keys()]
    ].rename(
        columns={
            c: new_c
            for c, new_c in columns_to_keep.items()
            if c in data_point_rows.columns
        }
    )
    if "vehicle_name" in data_point_rows.columns:
        data_point_rows["vehicle_name"] = data_point_rows["vehicle_name"].str.lower()
    if "wealth_quintile" in data_point_rows.columns:
        data_point_rows["wealth_quintile"] = (
            data_processing.recode_extraction_wealth_quintile(
                data_point_rows["wealth_quintile"]
            )
        )

    return data_point_rows

In [ ]:
# Totals should be fairly close to the mean of their child rows, just because in practice
# for our data here sex and wealth quintile are approximately evenly distributed, so the true
# (weighted) average is pretty close to the unweighted average.
# This is here mostly to catch egregious mistakes in data extraction.
def check_totals_reasonable(data_point_rows, definition_columns):
    total_markers = (
        data_point_rows[definition_columns].apply(lambda s: s.str.lower()) == "total"
    )
    num_total_markers = total_markers.sum(axis=1)
    total_rows = (total_markers).any(axis=1)
    for idx, total_row in data_point_rows[total_rows].iterrows():
        # Here we are traversing the imaginary tree created by the levels of detail.
        # The child rows are those that have one fewer "total" marker than the total row, and match the total row on all other definition columns.
        potential_child_rows = num_total_markers == num_total_markers[idx] - 1
        matching_rows = pd.Series(True, index=data_point_rows[potential_child_rows].index)
        for def_column in definition_columns:
            if str(total_row[def_column]).lower() != "total":
                matching_rows = matching_rows & (
                    data_point_rows[potential_child_rows][def_column] == total_row[def_column]
                )

        child_sets = total_markers[potential_child_rows][matching_rows].groupby(definition_columns)
        for _, child_set in child_sets:
            assert np.isclose(
                data_point_rows.loc[child_set.index].value.mean(),
                total_row["value"],
                atol=0,
                # This is not really based on anything; we can adjust it if we find that it is too strict
                rtol=0.1,
            )

In [ ]:
def save_results(
    country, sheet_name, sheet_data_needs, fortificant=None, vehicle=None, scenario=None
):
    sheet = pd.read_excel(
        "./Data Extraction Sheet.xlsx", sheet_name=f"{sheet_name} Extraction"
    )
    sheet = sheet[sheet.Country.str.lower() == country]

    if fortificant is not None:
        sheet = sheet[sheet.Fortificant.str.lower() == fortificant]

    if scenario is not None:
        # Strip out special characters and spaces
        sheet = sheet[
            sheet.Scenario.str.lower().str.replace("[\W_]+", "_", regex=True)
            == scenario
        ]

    if vehicle is not None:
        sheet = sheet[sheet.Vehicle.str.lower() == vehicle]

    assert len(sheet) > 0

    for need, short_need_name in sheet_data_needs.items():
        if need not in sheet["Data need"].values:
            # Needs filled by microdata
            print(f"Need {need} not extracted")
            continue

        print(f"Data need: {need}")
        need_rows = sheet[sheet["Data need"] == need]

        if country == "nigeria" and short_need_name == "vehicle_consumption/amount":
            # We do some interpolation here, see below
            continue

        if (
            country == "india"
            and short_need_name == "vehicle_consumption/fortifiability"
        ):
            # This is a special case with more processing to harmonize different source of information;
            # see below
            continue

        if (
            need.endswith("-- any")
            or short_need_name.endswith("_coverage")
            or short_need_name.endswith("effectiveness")
            or short_need_name == "vehicle_consumption/fortifiability"
        ):
            # Percentage
            assert (need_rows.Units == "%").all()
            assert (need_rows["Data point name"] == "percentage").all()
        elif need.endswith("-- amount") or short_need_name.endswith("amount"):
            # Consumption in g/day
            assert (need_rows.Units == "g/day").all()
            assert (
                need_rows["Data point name"].isin(["mean", "standard deviation"])
            ).all()
        elif need.endswith("-- amount among fortified") or short_need_name.endswith(
            "concentration"
        ):
            # Concentration in mcg/g
            assert (need_rows.Units == "mcg/g").all()
            assert (need_rows["Data point name"] == "concentration").all()
        else:
            raise ValueError()

        data_points = need_rows["Data point name"].unique()
        for data_point in data_points:
            data_point_rows = need_rows[need_rows["Data point name"] == data_point]
            data_point_rows = reformat_data(data_point_rows)

            definition_columns = [
                c for c in ["wealth_quintile", "sex"] if c in data_point_rows.columns
            ]

            data_point_rows = expand(data_point_rows, definition_columns)

            def groupby_apply(df, by, func):
                if len(by) > 0:
                    return df.groupby(by).apply(func)
                else:
                    return pd.Series([func(df)])

            check_totals_reasonable(data_point_rows, definition_columns)

            if len(definition_columns) > 0:
                data_point_rows = data_point_rows[
                    (
                        data_point_rows[definition_columns].apply(
                            lambda s: s.str.lower()
                        )
                        != "total"
                    ).all(axis=1)
                ]

            assert (
                groupby_apply(data_point_rows, definition_columns, lambda df: len(df))
                == 1
            ).all()

            if sheet_name == "Country-Vehicle" and "WRA" in need:
                assert (data_point_rows["sex"] == "Female").all()
                data_point_rows["age_start"] = 15
                data_point_rows["age_end"] = 50
            elif "U5" in need:
                data_point_rows["age_start"] = 0
                data_point_rows["age_end"] = 5

            if len(data_points) == 1:
                dir_name = short_need_name
            else:
                dir_name = f"{short_need_name}/{data_point_names[data_point]}"

            file_path = f"{results_dir}{('/' + fortificant) if fortificant is not None else ''}{('/' + vehicle) if vehicle is not None else ''}{('/' + scenario) if scenario is not None else ''}/{dir_name}/{country}.csv"
            pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
            data_point_rows.to_csv(file_path, index=False)

            if short_need_name == "baseline_fortification/any_coverage":
                # NOTE: For extractions, any == full coverage! We assume people are either fully
                # covered or not. This is probably reasonable in Nigeria -- why would people
                # buy multiple different types of bouillon?
                # We do something more sophisticated with India from microdata -- especially
                # relevant because lots of people have ration cards for a certain amount from one source.
                dir_name = "baseline_fortification/full_coverage"
                file_path = f"{results_dir}{('/' + fortificant) if fortificant is not None else ''}{('/' + vehicle) if vehicle is not None else ''}{('/' + scenario) if scenario is not None else ''}/{dir_name}/{country}.csv"
                pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
                data_point_rows.to_csv(file_path, index=False)

In [ ]:
for country, vehicle in config_utils.get_configured_combos(["location", "vehicle"]):
    save_results(
        country, "Country-Vehicle", data_needs["Country-Vehicle"], vehicle=vehicle
    )

for country, vehicle, fortificant in config_utils.get_configured_combos(
    ["location", "vehicle", "fortificant"]
):
    save_results(
        country,
        "Country-Vehicle-Fort",
        data_needs["Country-Vehicle-Fort"],
        fortificant=fortificant,
        vehicle=vehicle,
    )

for (
    country,
    vehicle,
    fortificant,
    intervention_scenario,
) in config_utils.get_configured_combos(
    ["location", "vehicle", "fortificant", "intervention_scenario"]
):
    save_results(
        country,
        "Scenario Definition",
        data_needs["Scenario Definition"],
        fortificant=fortificant,
        vehicle=vehicle,
        scenario=intervention_scenario,
    )

## Nigeria consumption interpolation/extrapolation

In [ ]:
sheet = pd.read_excel(
    "./Data Extraction Sheet.xlsx", sheet_name="Country-Vehicle Extraction"
)
sheet = sheet[sheet.Country.str.lower() == "nigeria"]

In [ ]:
def interpolate_extrapolate_consumption_amount(data):
    assert set(data["Data need"].unique()) == {
        "Vehicle consumption by WRA -- amount",
        "Vehicle consumption by U5 children -- amount",
    }
    assert (data["Country"] == "Nigeria").all()
    assert data["Vehicle"].nunique() == 1

    wra_data = data[
        data["Data need"] == "Vehicle consumption by WRA -- amount"
    ]
    assert (wra_data.Sex == "Female").all()
    u5_data = data[
        data["Data need"] == "Vehicle consumption by U5 children -- amount"
    ]
    assert set(wra_data["Data point name"].unique()) == {"mean", "standard deviation"}
    assert set(u5_data["Data point name"].unique()) == {"mean", "standard deviation"}

    wra_mean_data = expand(reformat_data(wra_data[wra_data["Data point name"] == "mean"]), ["wealth_quintile", "sex"])
    u5_mean_data = expand(reformat_data(u5_data[u5_data["Data point name"] == "mean"]), ["wealth_quintile", "sex"])
    wra_sd_data = expand(reformat_data(wra_data[wra_data["Data point name"] == "standard deviation"]), ["wealth_quintile", "sex"])
    u5_sd_data = expand(reformat_data(u5_data[u5_data["Data point name"] == "standard deviation"]), ["wealth_quintile", "sex"])

    check_totals_reasonable(wra_mean_data, ["wealth_quintile", "sex"])
    check_totals_reasonable(u5_mean_data, ["wealth_quintile", "sex"])
    # NOTE: SD does not aggregate up to totals the same way!

    # Due to the specific tabulations in the NFCMS 2021 report, we don't have wealth disaggregation for children
    assert (u5_mean_data["wealth_quintile"] == 'Total').all()
    assert (u5_sd_data["wealth_quintile"] == 'Total').all()
    # So we will use the wealth pattern from WRA to split up the U5 data
    mean_wealth_gradient = wra_mean_data[wra_mean_data.wealth_quintile.astype(str).str.lower() != "total"][["wealth_quintile", "value"]]
    assert mean_wealth_gradient["wealth_quintile"].nunique() == 5
    print('WRA mean wealth gradient:')
    display(mean_wealth_gradient)

    u5_wealth_quintile_probabilities = (
        pd.read_csv('../results/wealth_quintile_probabilities/nigeria.csv')
            .pipe(lambda df: df[(df.age_start == 0) & (df.age_end == 5)])
            .drop(columns=['age_start', 'age_end', 'pregnant'])
            .set_index("sex")
            .rename_axis(columns="wealth_quintile")
            .rename(columns=lambda col: int(col))
            .stack()
    )
    assert np.allclose(u5_wealth_quintile_probabilities.groupby("sex").sum(), 1)
    print('U5 wealth quintile probabilities:')
    display(u5_wealth_quintile_probabilities)

    # FIXME: This operation is essentially the same as _distribute_by_disparities_multiplicative in the pregnancy sim's
    # loader.py (and countless other places distributing by a pattern has been implemented).
    unscaled_aggregates = (
        u5_wealth_quintile_probabilities * mean_wealth_gradient.set_index("wealth_quintile").value
    ).groupby(level=["sex"]).sum()
    print('WRA gradient aggregated by U5 wealth quintile probabilities:')
    display(unscaled_aggregates)

    u5_means_by_sex = u5_mean_data[u5_mean_data.sex.astype(str).str.lower() != "total"].set_index("sex").value
    print('U5 means by sex:')
    display(u5_means_by_sex)

    scaling_factors = u5_means_by_sex / unscaled_aggregates
    print('Scaling factors to apply to WRA wealth gradient to match U5 means:')
    display(scaling_factors)

    detailed_u5_mean_data = (
        mean_wealth_gradient
            .merge(scaling_factors.rename("scalar").reset_index(), how="cross")
            .assign(value=lambda df: df.value * df.scalar, vehicle_name=u5_mean_data.vehicle_name.iloc[0])
            .drop(columns="scalar")
            [u5_mean_data.columns]
    )
    check_totals_reasonable(pd.concat([u5_mean_data, detailed_u5_mean_data], ignore_index=True), ["wealth_quintile", "sex"])
    print('Result after distribution by WRA gradient:')
    display(detailed_u5_mean_data)

    # Now the standard deviations are even trickier.
    sd_wealth_gradient = wra_sd_data[wra_sd_data.wealth_quintile.astype(str).str.lower() != "total"][["wealth_quintile", "value"]]
    assert sd_wealth_gradient["wealth_quintile"].nunique() == 5
    print('WRA SD wealth gradient:')
    display(sd_wealth_gradient)

    unscaled_aggregates = (
        u5_wealth_quintile_probabilities * sd_wealth_gradient.set_index("wealth_quintile").value
    ).groupby(level=["sex"]).sum()
    print('WRA SD gradient aggregated by U5 wealth quintile probabilities, aka E[Var(consumption)|wealth]:')
    display(unscaled_aggregates)

    print('U5 means by sex:')
    display(u5_means_by_sex)

    quintile_deviations = detailed_u5_mean_data.set_index(["sex", "wealth_quintile"]).value - u5_means_by_sex
    print('Quintile deviations from U5 means by sex:')
    display(quintile_deviations)

    u5_between_group_variance = (quintile_deviations**2 * u5_wealth_quintile_probabilities).groupby("sex").sum()
    print('U5 between-group variance by sex, aka Var(E[consumption|wealth]):')
    display(u5_between_group_variance)

    u5_total_variance_by_sex = u5_sd_data[u5_sd_data.sex.astype(str).str.lower() != "total"].set_index("sex").value**2
    print('U5 total variance by sex, aka Var(consumption):')
    display(u5_total_variance_by_sex)

    u5_within_group_variance = u5_total_variance_by_sex - u5_between_group_variance
    print('Decomposition of variance, where between-group is Var(E[consumption|wealth]) and within-group is E[Var(consumption|wealth)]:')
    display(pd.DataFrame({
        "total_variance": u5_total_variance_by_sex,
        "between_group_variance": u5_between_group_variance,
        "within_group_variance": u5_within_group_variance,
    }))

    scaling_factors = np.sqrt(u5_within_group_variance) / unscaled_aggregates
    print('Scaling factors to apply to WRA SD wealth gradient to match U5 within-group variance:')
    display(scaling_factors)

    detailed_u5_sd_data = (
        sd_wealth_gradient
            .merge(scaling_factors.rename("scalar").reset_index(), how="cross")
            .assign(value=lambda df: df.value * df.scalar, vehicle_name=u5_sd_data.vehicle_name.iloc[0])
            .drop(columns="scalar")
            [u5_sd_data.columns]
    )
    print('Result after distribution by WRA SD gradient:')
    display(detailed_u5_sd_data)

    result = []

    for statistic in ["mean", "standard deviation"]:
        print(f"Interpolating and extrapolating {statistic}...")
        if statistic == "mean":
            detailed_u5_data = detailed_u5_mean_data
            detailed_wra_data = wra_mean_data[wra_mean_data.wealth_quintile.astype(str).str.lower() != "total"]
        elif statistic == "standard deviation":
            detailed_u5_data = detailed_u5_sd_data
            detailed_wra_data = wra_sd_data[wra_sd_data.wealth_quintile.astype(str).str.lower() != "total"]

        u5_rows = (
            detailed_u5_data
                .assign(age_start=0, age_end=5)
                .set_index(["sex", "age_start", "age_end", "wealth_quintile", "vehicle_name"])
                .value
        )

        male_to_female_ratio = u5_rows[
            u5_rows.index.get_level_values("sex") == "Male"
        ].droplevel("sex") / u5_rows[
            u5_rows.index.get_level_values("sex") == "Female"
        ].droplevel(
            "sex"
        )
        assert np.allclose(male_to_female_ratio, male_to_female_ratio.mean())
        male_to_female_ratio = male_to_female_ratio.mean()
        print(f"Male to female ratio: {male_to_female_ratio}")

        adult_rows = (
            pd.concat(
                [
                    detailed_wra_data,
                    detailed_wra_data.assign(sex="Male", value=lambda df: df.value * male_to_female_ratio),
                ],
                ignore_index=True,
            )
            .assign(age_start=15, age_end=125)
            .set_index(["sex", "age_start", "age_end", "wealth_quintile", "vehicle_name"])
            .value
        )

        adolescent_rows = (
            (
                adult_rows.droplevel(["age_start", "age_end"]) * 0.5
                + u5_rows.droplevel(["age_start", "age_end"]) * 0.5
            )
            .reset_index()
            .assign(age_start=5, age_end=15)
            .set_index(["sex", "age_start", "age_end", "wealth_quintile", "vehicle_name"])
            .value
        )

        result.append(pd.concat(
            [
                u5_rows,
                adolescent_rows,
                adult_rows,
            ]
        ).sort_index())

    assert len(result) == 2
    return tuple(result)

In [ ]:
for location, vehicle in config_utils.get_configured_combos(["location", "vehicle"]):
    if location != "nigeria":
        continue

    print(vehicle)

    mean_consumption, sd_consumption = interpolate_extrapolate_consumption_amount(
        sheet[
            (sheet["Vehicle"].str.lower() == vehicle) &
            (sheet["Data need"].str.endswith("-- amount"))
        ]
    )
    print(f"Mean consumption for {vehicle}:")
    display(mean_consumption)
    print(f"SD consumption for {vehicle}:")
    display(sd_consumption)

    file_path = f"{results_dir}/{vehicle}/vehicle_consumption/amount/mean/nigeria.csv"
    pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
    mean_consumption.to_csv(file_path)

    file_path = f"{results_dir}/{vehicle}/vehicle_consumption/amount/sd/nigeria.csv"
    pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
    sd_consumption.to_csv(file_path)

# Partial coverage amount for Ethiopia and Nigeria

0, since there is no partial coverage in our baseline scenarios here.

In [ ]:
for location, vehicle, fortificant in config_utils.get_configured_combos(
    ["location", "vehicle", "fortificant"]
):
    if location == "india":
        continue
    df = pd.DataFrame(
        {
            "wealth_quintile": data_processing.WEALTH_QUINTILES,
            "vehicle_name": vehicle,
            "value": 0,
        }
    )
    for quantity in data_point_names.values():
        path = f"{results_dir}/{fortificant}/{vehicle}/baseline_fortification/partial_coverage_amount/{quantity}/{location}.csv"
        pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
        print(path)
        df.to_csv(path, index=False)

## India industry consolidation

In [ ]:
# We apply the India industry consolidation (overall fortifiability from the extraction sheet) to the rice *not
# distributed by the government*.
# First we disaggregate the industry consolidation number by wealth according to a proxy from HCES: the proportion purchased.

sheet = pd.read_excel(
    "./Data Extraction Sheet.xlsx", sheet_name="Country-Vehicle Extraction"
)
sheet = sheet[sheet.Country.str.lower() == "india"]
assert (sheet.Vehicle.str.lower() == "rice").all()

overall_fortifiability_row = sheet[
    sheet["Data need"]
    == 'Vehicle "fortifiability" (essentially amount industrially produced)'
]
assert len(overall_fortifiability_row) == 1
assert (overall_fortifiability_row.Quintile == "Total").all()

industry_consolidation = float(overall_fortifiability_row.Value.iloc[0])
industry_consolidation

In [ ]:
disparity_proxy = (
    pd.read_csv("../hces/india_rice_fortifiability_disparities.csv")
    .set_index(["sex", "age_start", "age_end", "wealth_quintile"])
    .value
)
disparity_proxy

In [ ]:
# Equally weighting the quintiles, which isn't quite right but close
industry_consolidation_by_quintile = (
    industry_consolidation * disparity_proxy
) / disparity_proxy.mean()
industry_consolidation_by_quintile

In [ ]:
government_rice = (
    pd.read_csv("../hces/india_proportion_government_rice.csv")
    .set_index(["sex", "age_start", "age_end", "wealth_quintile"])
    .value
)
government_rice

In [ ]:
# We assume all rice distributed by the government is fortifiable.
# The industry consolidation applies to non-government distributed rice.
fortifiability_by_quintile = (
    government_rice + (1 - government_rice) * industry_consolidation_by_quintile
)
fortifiability_by_quintile

In [ ]:
file_path = f"{results_dir}/rice/vehicle_consumption/fortifiability/india.csv"
pathlib.Path(file_path).parent.mkdir(parents=True, exist_ok=True)
fortifiability_by_quintile.reset_index().assign(vehicle_name="rice").to_csv(
    file_path, index=False
)